# 01: データ前処理

CSVデータの読み込みとクリーニング、基本統計情報の確認を行います。

## 目的
- 過去当選番号データの読み込み
- データの基本統計情報の確認
- 学習用データセットの準備（直近100回分）

## 基準回号
- **基準回号**: 第6758回（2025年6月30日）
- この回号を基準にさかのぼって学習データを構築します


In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from typing import Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

# プロジェクトルートのパスを設定
PROJECT_ROOT = Path(__file__).parent.parent if '__file__' in globals() else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'

print(f"プロジェクトルート: {PROJECT_ROOT}")
print(f"データディレクトリ: {DATA_DIR}")


In [ ]:
# CSVデータの読み込み
csv_path = DATA_DIR / 'past_results.csv'

if not csv_path.exists():
    raise FileNotFoundError(f"データファイルが見つかりません: {csv_path}")

df = pd.read_csv(csv_path)

# 当選番号を文字列型に変換（数値型で読み込まれる可能性があるため）
df['n3_winning'] = df['n3_winning'].astype(str).str.replace('.0', '', regex=False)
df['n4_winning'] = df['n4_winning'].astype(str).str.replace('.0', '', regex=False)

print(f"データ行数: {len(df)}")
print(f"\nデータの最初の5行:")
print(df.head())
print(f"\nデータの基本情報:")
print(df.info())


In [ ]:
# データクリーニング
# 1. 必須フィールドの確認
required_columns = ['round_number', 'draw_date', 'n3_winning', 'n4_winning']
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"必須カラムが見つかりません: {missing_columns}")

# 2. NULL値の処理
df = df.dropna(subset=required_columns)

# 3. 回号の検証
df = df[df['round_number'].between(1, 9999)]

# 4. 当選番号のフォーマット検証
df = df[df['n3_winning'].str.match(r'^\d{3}$', na=False)]
df = df[df['n4_winning'].str.match(r'^\d{4}$', na=False)]

# 5. 回号の降順でソート（最新が先頭）
df = df.sort_values('round_number', ascending=False).reset_index(drop=True)

print(f"クリーニング後のデータ行数: {len(df)}")
print(f"\n回号の範囲: {df['round_number'].min()} 〜 {df['round_number'].max()}")


In [ ]:
# 基準回号の確認
BASE_ROUND = 6758
BASE_DATE = '2025-06-30'

base_row = df[df['round_number'] == BASE_ROUND]
if len(base_row) == 0:
    raise ValueError(f"基準回号 {BASE_ROUND} が見つかりません")

print(f"基準回号: {BASE_ROUND}")
print(f"基準日: {BASE_DATE}")
print(f"\n基準回号のデータ:")
print(base_row[['round_number', 'draw_date', 'n3_winning', 'n4_winning']])


In [ ]:
# 学習用データセットの準備（直近100回分）
# 基準回号（6758）からさかのぼって100回分を取得

# 基準回号のインデックスを取得
base_idx = df[df['round_number'] == BASE_ROUND].index[0]

# 基準回号を含む100回分を取得
train_df = df.iloc[base_idx:base_idx+100].copy()

# 回号の昇順でソート（古い順）
train_df = train_df.sort_values('round_number', ascending=True).reset_index(drop=True)

print(f"学習用データセット: {len(train_df)}回分")
print(f"回号範囲: {train_df['round_number'].min()} 〜 {train_df['round_number'].max()}")
print(f"\n最初の5行:")
print(train_df[['round_number', 'draw_date', 'n3_winning', 'n4_winning']].head())
print(f"\n最後の5行:")
print(train_df[['round_number', 'draw_date', 'n3_winning', 'n4_winning']].tail())


In [ ]:
# データの基本統計情報
print("=== N3当選番号の統計 ===")
n3_digits = train_df['n3_winning'].str.split('', expand=True).iloc[:, 1:4].astype(int)
print(f"各桁の平均値:")
print(n3_digits.mean())
print(f"\n各桁の出現頻度（0-9）:")
for i in range(3):
    col_idx = i + 1  # split結果の列インデックス（1, 2, 3）
    print(f"  桁{i+1}:")
    print(n3_digits.iloc[:, i].value_counts().sort_index())

print("\n=== N4当選番号の統計 ===")
n4_digits = train_df['n4_winning'].str.split('', expand=True).iloc[:, 1:5].astype(int)
print(f"各桁の平均値:")
print(n4_digits.mean())
print(f"\n各桁の出現頻度（0-9）:")
for i in range(4):
    col_idx = i + 1  # split結果の列インデックス（1, 2, 3, 4）
    print(f"  桁{i+1}:")
    print(n4_digits.iloc[:, i].value_counts().sort_index())


In [ ]:
# 罫線マスターデータの読み込み確認
keisen_path = DATA_DIR / 'keisen_master.json'

if not keisen_path.exists():
    raise FileNotFoundError(f"罫線マスターファイルが見つかりません: {keisen_path}")

with open(keisen_path, 'r', encoding='utf-8') as f:
    keisen_master = json.load(f)

print("罫線マスターデータの構造:")
print(f"  N3: {list(keisen_master.get('n3', {}).keys())}")
print(f"  N4: {list(keisen_master.get('n4', {}).keys())}")

# サンプルデータの確認
if 'n3' in keisen_master and '百の位' in keisen_master['n3']:
    sample_key = list(keisen_master['n3']['百の位'].keys())[0]
    sample_value = keisen_master['n3']['百の位'][sample_key]
    print(f"\nサンプル（N3百の位、前々回={sample_key}）:")
    print(f"  キー数: {len(sample_value)}")
    if sample_value:
        first_key = list(sample_value.keys())[0]
        print(f"  前回={first_key}の場合の予測出目: {sample_value[first_key]}")


In [ ]:
# データセットの保存（次のNotebookで使用）
# 学習用データセットをCSV形式で保存
train_csv_path = DATA_DIR / 'train_data_100.csv'
train_df.to_csv(train_csv_path, index=False)
print(f"学習用データセットを保存しました: {train_csv_path}")
print(f"ファイルサイズ: {train_csv_path.stat().st_size / 1024:.2f} KB")


## 次のステップ

データ前処理が完了しました。次のNotebook (`02_chart_generation.ipynb`) で、各回号に対して4パターン（A1/A2/B1/B2）の予測表を生成します。
